# Owner Name Entity Resolution Pipeline (No-Embedding / Fuzzy-Only)

Cluster similar owner names, minimize false positives - **without any embedding model**, so **no cosine similarity**.

| Item                     | Value                                                                                  |
| ------------------------ | -------------------------------------------------------------------------------------- |
| Dataset                  | `temp_unique_corporate_names` (~349,413 unique names)                                  |
| Embedding model / cosine | **removed**                                                                            |
| Candidate search         | **Token blocking** (inverted index on discriminative tokens)                           |
| Scoring                  | **Streaming** (generate + feature + decide in one pass - never materializes all pairs) |
| Matching                 | Fuzzy string metrics (RapidFuzz + Levenshtein)                                         |
| Clustering               | Union-Find (Disjoint Set)                                                              |

> **Why streaming:** single-token blocking on 349k names produces tens of millions of candidate pairs.
> Materializing all pairs + a full feature table (with string columns) exhausts RAM and kills the kernel.
> Instead we stream: for each candidate pair we compute features and decide immediately, keeping only the
> (few) accepted edges plus a capped sample for inspection. Memory stays flat.

> Notebook is code-only scaffold. Cells are NOT executed.


## Step 0 : Setup & Configuration


In [ ]:
# Step 0.1 : Install dependencies (run once, uncomment) - no torch / sentence-transformers / faiss
# !pip install pandas numpy pyarrow rapidfuzz python-Levenshtein tqdm

In [ ]:
# Step 0.2 : Imports
import os
import re
import unicodedata
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein

In [ ]:
# Step 0.3 : Global config / paths
CONFIG = {
    "input_file":    "temp_unique_corporate_names",
    "input_col":     "owner_name_cleaned",
    "artifacts_dir": "artifacts_noemb",

    # ---- Blocking / streaming ----
    "max_block":     400,      # skip tokens in more than this many names (combinatorial blow-up + weak signal)
    "common_df":     40,       # a token in >= this many names is a WEAK discriminator (UNIVERSAL, SHREE, GEETA);
                               # sharing only weak tokens must NOT merge (prevents Union-Find chaining)
    "prefilter_tss": 60,       # skip a pair whose token_set_ratio is below this (cheap pre-cut, before full features)
    "sample_cap":    300_000,  # max rows kept in the inspection feature table (streaming keeps only a sample)

    # ---- Rule engine thresholds (no cosine) ----
    "rules": {
        "core_sim_min":   85,
        "near_exact_rf":  95,
        "r1_token_set":   95, "r1_rapidfuzz": 90,
        "r2_lev":         2,  "r2_rapidfuzz": 90,
        "r3_core_sim":    90, "r3_token_set": 85,
    },
}

os.makedirs(CONFIG["artifacts_dir"], exist_ok=True)
def art(name):
    return os.path.join(CONFIG["artifacts_dir"], name)
CONFIG

## Phase 1 : Data Cleaning

Uppercase, `&`->AND, punctuation->space, unicode strip, collapse spaces.


In [ ]:
# Step 1.1 : Load raw data
df = pd.read_csv(CONFIG["input_file"])
df = df.rename(columns={CONFIG["input_col"]: "original_name"})
df["original_name"] = df["original_name"].astype(str)
print("rows:", len(df))
df.head(10)

In [ ]:
# Step 1.2 : Cleaning function
_PUNCT_TO_SPACE = re.compile(r"[^A-Z0-9&\s]")
_MULTISPACE     = re.compile(r"\s+")

def clean_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    s = unicodedata.normalize("NFKD", name)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = s.upper()
    s = s.replace("&", " AND ")
    s = _PUNCT_TO_SPACE.sub(" ", s)
    s = _MULTISPACE.sub(" ", s).strip()
    return s

In [ ]:
# Step 1.3 : Apply cleaning
df["clean_name"] = df["original_name"].map(clean_name)
df[["original_name", "clean_name"]].head(10)

In [ ]:
# Step 1.4 : Validation - inspect 100 random records
for o, c in df.sample(100, random_state=42)[["original_name", "clean_name"]].itertuples(index=False):
    print(f"{o!r:60} -> {c!r}")

## Phase 2 : Typo / OCR Normalization

Token-level correction of common business-word OCR errors.


In [ ]:
# Step 2.1 : OCR correction dictionary (business words only)
OCR_DICT = {
    "TRAVLES": "TRAVELS", "TRAVELES": "TRAVELS",
    "LIMITD": "LIMITED", "LIMTED": "LIMITED", "LIMITEDD": "LIMITED",
    "PRIVATELIMITED": "PRIVATE LIMITED",
    "COMPNY": "COMPANY", "COMPANYY": "COMPANY",
    "COOPERATIV": "COOPERATIVE",
    "INFRASTRUOTURE": "INFRASTRUCTURE", "INFRASTRUCTUR": "INFRASTRUCTURE",
    "ENTERPRISS": "ENTERPRISES", "ENTERPRISESS": "ENTERPRISES",
    "CONSTRUCTON": "CONSTRUCTION", "CONSTUCTION": "CONSTRUCTION", "CONSTRUTION": "CONSTRUCTION",
    "INDUSTRIS": "INDUSTRIES", "SERVICS": "SERVICES",
}

In [ ]:
# Step 2.2 : Token-level replacement
def normalize_typos(clean: str) -> str:
    return " ".join(OCR_DICT.get(t, t) for t in clean.split())
df["typo_fixed"] = df["clean_name"].map(normalize_typos)

In [ ]:
# Step 2.3 : Validation - show only changed rows (up to 200)
changed = df[df["clean_name"] != df["typo_fixed"]][["clean_name", "typo_fixed"]].head(200)
print("changed rows:", (df["clean_name"] != df["typo_fixed"]).sum())
for a, b in changed.itertuples(index=False):
    print(f"{a!r:60} -> {b!r}")

## Phase 3 : Canonical Name Generation

Strip legal suffixes, keep industry descriptors.


In [ ]:
# Step 3.1 : Legal / filler stopwords
LEGAL_STOPWORDS = {
    "PRIVATE", "LIMITED", "PVT", "LTD", "LLP", "PLC",
    "AND", "COMPANY", "CO", "CORPORATION", "CORP", "INC", "THE",
}

In [ ]:
# Step 3.2 : Canonicalization
def canonicalize(name: str) -> str:
    toks = [t for t in name.split() if t not in LEGAL_STOPWORDS]
    canon = " ".join(toks).strip()
    return canon if canon else name
df["canonical_name"] = df["typo_fixed"].map(canonicalize)
df[["original_name", "clean_name", "canonical_name"]].head(10)

In [ ]:
# Step 3.3 : Validation - original -> canonical for 200 random rows
for o, c in df.sample(200, random_state=7)[["original_name", "canonical_name"]].itertuples(index=False):
    print(f"{o!r:60} -> {c!r}")

In [ ]:
# Step 3.4 : Persist + stable id + drop empty canonicals
df = df[df["canonical_name"].str.len() > 0].reset_index(drop=True)
df["row_id"] = np.arange(len(df))
df.to_parquet(art("names_clean.parquet"), index=False)
print("kept rows:", len(df))

## Phase 4 : Token Helpers

Discriminative-token machinery that drives both blocking and features.


In [ ]:
# Step 4.1 : Generic/business + geo keywords (do NOT identify a company on their own)
BUSINESS_KEYWORDS = {
    "TOURS", "TRAVELS", "INFRASTRUCTURE", "INFRA", "INFRATECH", "TECH",
    "TECHNOLOGY", "TECHNOLOGIES", "CONSTRUCTION", "CONSTRUCTIONS",
    "ENTERPRISE", "ENTERPRISES", "INDUSTRY", "INDUSTRIES", "SERVICE", "SERVICES",
    "AGENCY", "AGENCIES", "TRADING", "TRADER", "TRADERS", "TRADE",
    "PROJECT", "PROJECTS", "FOOD", "FOODS", "AUTOMOBILE", "AUTOMOBILES", "AUTO",
    "FINANCE", "FIN", "ENGINEERING", "WORK", "WORKS", "FARM", "FARMS", "MART",
    "BANK", "JEWELLER", "JEWELLERS", "JEWELLERY", "STEEL", "STEELS",
    "LOGISTIC", "LOGISTICS", "TRANSPORT", "TRANSPORTS", "SOLUTION", "SOLUTIONS",
    "NETWORK", "NETWORKS", "CONSULTING", "CONSULTANCY", "POWER", "MARINE",
    "ELECTRIC", "LIFESCIENCES", "SCIENCE", "SCIENCES", "LIFE", "INVESTMENT",
    "INVESTMENTS", "COMMUNICATION", "COMMUNICATIONS", "COFFEE", "SPONGE",
    "INDIA", "INDIAN", "BHARAT",
}

def _fold(tok):
    return tok[:-1] if len(tok) > 4 and tok.endswith("S") else tok

def core_tokens(name):
    # discriminative tokens: drop generic words AND single-char initials
    return {_fold(t) for t in name.split() if t not in BUSINESS_KEYWORDS and len(t) > 1}

def strip_business(name):
    return " ".join(t for t in name.split() if t not in BUSINESS_KEYWORDS)

def is_single_token(name):
    return len(name.split()) == 1

## Phase 5 : Inverted Index (blocking structure)

`discriminative_token -> [row_ids]`. Two names are candidates only if they share such a token.
Also precompute each row's core-token set once (reused for dedup + features - avoids recomputing per pair).


In [ ]:
# Step 5.1 : Build inverted index + per-row core-token sets
canon = df["canonical_name"].to_numpy()

core_sets = [core_tokens(c) for c in tqdm(canon, desc="core_sets")]   # list of sets, ~one per row

inverted = defaultdict(list)
for idx, cs in enumerate(core_sets):
    for tok in cs:
        inverted[tok].append(idx)

print("distinct discriminative tokens:", len(inverted))

In [ ]:
# Step 5.2 : Report block sizes (candidate volume comes from here)
blk = pd.Series({t: len(ids) for t, ids in inverted.items()})
print("blocks total       :", len(blk))
print("blocks over max    :", int((blk > CONFIG["max_block"]).sum()), "(skipped in scoring)")
print("largest 15 blocks  :")
print(blk.sort_values(ascending=False).head(15))

## Phase 6 : Feature Engineering (no cosine)

Plan's features minus cosine + the anti-over-merge features (core_shared, core_sim, single_token).
`core_a` / `core_b` are passed in so we reuse the precomputed sets instead of recomputing.


In [ ]:
# Step 6.1 : Per-pair feature computation (no cosine)
def pair_features(a, b, core_a, core_b):
    ta, tb = set(a.split()), set(b.split())
    inter, union = ta & tb, ta | tb
    core_inter = core_a & core_b

    sa, sb = strip_business(a), strip_business(b)
    core_sim = max(
        fuzz.token_set_ratio(sa, sb),
        fuzz.ratio(sa.replace(" ", ""), sb.replace(" ", "")),   # catches concatenation
    )
    return {
        "rapidfuzz":     fuzz.ratio(a, b),
        "token_sort":    fuzz.token_sort_ratio(a, b),
        "token_set":     fuzz.token_set_ratio(a, b),
        "levenshtein":   Levenshtein.distance(a, b),
        "jaccard":       (len(inter) / len(union)) if union else 0.0,
        "common_tokens": len(inter),
        "prefix_match":  int(a[:3] == b[:3]),
        "biz_kw_match":  int(len(inter) > 0 and len(core_inter) == 0),
        "core_shared":   len(core_inter),
        # strong_shared = shared discriminative tokens that are also RARE (not corpus-common).
        # COMMON_TOKENS is built from the corpus in Step 8.1 (before scoring).
        "strong_shared": len(core_inter - COMMON_TOKENS),
        "core_sim":      core_sim,
        "single_token":  int(is_single_token(a) or is_single_token(b)),
    }

## Phase 7 : Rule Engine (no cosine)

**Hard rejects:** (a) single-token names merge only if identical; (b) `core_sim < 85`; (c) `strong_shared == 0`
(no shared _rare_ token - sharing only a common brand word like UNIVERSAL/SHREE is not enough; this stops
Union-Find from chaining thousands of distinct firms into one blob).
**Merge:** near-exact `rf>=95`; Rule 1 `token_set>=95 AND rf>=90`; Rule 2 `lev<=2 AND rf>=90`; Rule 3 `core_sim>=90 AND token_set>=85`.


In [ ]:
# Step 7.1 : Decision function (no cosine)
R = CONFIG["rules"]

def decide(f) -> bool:
    if f["single_token"] and f["levenshtein"] != 0:
        return False
    if f["rapidfuzz"] >= R["near_exact_rf"]:
        return True
    if f["core_sim"] < R["core_sim_min"]:
        return False
    # must share at least one RARE discriminative token. Sharing only a common brand word
    # (UNIVERSAL, SHREE, GLOBAL) is NOT enough -> stops distinct firms chaining into one blob.
    if f["strong_shared"] == 0:
        return False
    if f["token_set"] >= R["r1_token_set"] and f["rapidfuzz"] >= R["r1_rapidfuzz"]:
        return True
    if f["levenshtein"] <= R["r2_lev"] and f["rapidfuzz"] >= R["r2_rapidfuzz"]:
        return True
    if f["core_sim"] >= R["r3_core_sim"] and f["token_set"] >= R["r3_token_set"]:
        return True
    return False

## Phase 8 : Streaming Candidate Scoring

The memory-safe core. One pass over the inverted index:

- **8.1** builds the block plan (skip oversized blocks, logged - never silent).
- **8.2** streams every candidate pair: dedup (each pair owned by its smallest shared token, so no giant
  `seen` set), cheap prefilter, full features, `decide`. Keeps only **accepted edges** (small) plus a
  **capped feature sample** for inspection/eval. All candidate pairs are never held in memory at once.


In [ ]:
# Step 8.1 : Block plan (skip oversized blocks; report dropped coverage)
MAXB, PREF, CAP = CONFIG["max_block"], CONFIG["prefilter_tss"], CONFIG["sample_cap"]

# COMMON_TOKENS: weak discriminators (appear in many names). Sharing only these must not merge.
COMMON_TOKENS = {t for t, ids in inverted.items() if len(ids) >= CONFIG["common_df"]}
print(f"common (weak) tokens >= {CONFIG['common_df']} names: {len(COMMON_TOKENS)}",
      "e.g.", sorted(COMMON_TOKENS, key=lambda t: -len(inverted[t]))[:12])

blocks  = {t: ids for t, ids in inverted.items() if len(ids) <= MAXB}
dropped = {t: len(ids) for t, ids in inverted.items() if len(ids) > MAXB}
if dropped:
    print(f"skipping {len(dropped)} oversized blocks (> {MAXB}). Pairs whose only tie is such a token are")
    print("NOT compared -> recall cost. Largest dropped:", sorted(dropped.items(), key=lambda x: -x[1])[:10])
print("blocks to score:", len(blocks))

In [ ]:
# Step 8.2 : Stream pairs -> accepted edges + capped inspection sample
accepted_edges = []          # (i, j) of merged pairs  -> small, only true merges
sample_rows    = []          # capped feature rows for inspection/eval
n_seen = n_pref = n_acc = 0  # counters

for tok, ids in tqdm(blocks.items(), desc="score"):
    for a in range(len(ids)):
        ia = ids[a]; ca = core_sets[ia]; na = canon[ia]
        for b in range(a + 1, len(ids)):
            ib = ids[b]
            lo, hi = (ia, ib) if ia < ib else (ib, ia)
            clo, chi = core_sets[lo], core_sets[hi]

            # dedup WITHOUT a global set: process a pair only in the block of its
            # smallest shared discriminative token. min() over a small set is cheap.
            shared = clo & chi
            if min(shared) != tok:
                continue
            n_seen += 1

            nlo, nhi = canon[lo], canon[hi]
            # cheap prefilter before the full (costlier) feature computation
            if fuzz.token_set_ratio(nlo, nhi) < PREF:
                continue
            n_pref += 1

            f = pair_features(nlo, nhi, clo, chi)
            m = decide(f)
            if m:
                accepted_edges.append((lo, hi))
                n_acc += 1
            # keep a capped sample (both accepted + rejected) for inspection / eval
            if len(sample_rows) < CAP:
                sample_rows.append({"i": lo, "j": hi, "owner1": nlo, "owner2": nhi,
                                    "merge": m, **f})

print(f"pairs scored: {n_seen:,} | passed prefilter: {n_pref:,} | accepted: {n_acc:,}")
print(f"sample kept : {len(sample_rows):,} (cap {CAP:,})")

In [ ]:
# Step 8.3 : Materialize accepted edges + inspection sample (both small)
accepted = pd.DataFrame(accepted_edges, columns=["i", "j"])
accepted.to_parquet(art("accepted_pairs.parquet"), index=False)

features_sample = pd.DataFrame(sample_rows)   # for validation + evaluation only
print("accepted pairs:", len(accepted), "| sample rows:", len(features_sample))

In [ ]:
# Step 8.4 : Validation - inspect accepted + rejected from the sample
acc_s = features_sample[features_sample["merge"]]
rej_s = features_sample[~features_sample["merge"]]
print("===== ACCEPTED sample =====")
display(acc_s.sample(min(1000, len(acc_s)), random_state=2)[
    ["owner1", "owner2", "token_set", "rapidfuzz", "levenshtein", "core_sim"]])
print("===== REJECTED sample =====")
display(rej_s.sample(min(1000, len(rej_s)), random_state=3)[
    ["owner1", "owner2", "token_set", "rapidfuzz", "levenshtein", "core_sim", "core_shared", "single_token"]])

## Phase 9 : Union-Find (Disjoint Set)

Merge accepted edges. `union` is idempotent, so duplicate edges (none here, but safe) cost nothing.


In [ ]:
# Step 9.1 : Union-Find with path compression + union by rank
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

In [ ]:
# Step 9.2 : Merge accepted pairs
uf = UnionFind(len(df))
for i, j in accepted[["i", "j"]].itertuples(index=False):
    uf.union(int(i), int(j))
df["cluster_id"] = [uf.find(x) for x in range(len(df))]
df[["canonical_name", "cluster_id"]].head(10)

In [ ]:
# Step 9.3 : Validation - cluster size distribution
sizes = df["cluster_id"].value_counts()
print("total clusters :", sizes.shape[0])
print("singletons     :", (sizes == 1).sum())
print("avg size       :", round(sizes.mean(), 3))
print("largest 10     :\n", sizes.head(10))

In [ ]:
# Step 9.4 : Inspect clusters with > 20 members (over-merge check)
for cid in list(sizes[sizes > 20].index)[:20]:
    members = df.loc[df.cluster_id == cid, "canonical_name"].tolist()
    print(f"\n--- cluster {cid} ({len(members)}) ---")
    for m in members[:30]:
        print("   ", m)

## Phase 10 : Canonical Owner Name Selection

Most frequent original name per cluster, tie-break longest.


In [ ]:
# Step 10.1 : Choose canonical owner per cluster
def pick_canonical(group: pd.DataFrame) -> str:
    counts = group["original_name"].value_counts()
    top = counts[counts == counts.max()].index
    return max(top, key=len)

cluster_canon = (
    df.groupby("cluster_id")
      .apply(pick_canonical)
      .rename("canonical_owner")
      .reset_index()
)
# cluster_size = # names in the cluster. Map on cluster_canon's OWN cluster_id column (index-safe).
cluster_canon["cluster_size"] = cluster_canon["cluster_id"].map(df["cluster_id"].value_counts())

In [ ]:
# Step 10.2 : Attach canonical owner to every row + persist
df = df.merge(cluster_canon[["cluster_id", "canonical_owner"]], on="cluster_id", how="left")
df.to_parquet(art("names_clustered.parquet"), index=False)
cluster_canon.to_parquet(art("cluster_canonical.parquet"), index=False)
cluster_canon.sort_values("cluster_size", ascending=False).head(20)

In [ ]:
# Step 10.3 : Validation - inspect largest 100 clusters
for cid, owner, sz in cluster_canon.sort_values("cluster_size", ascending=False).head(100).itertuples(index=False):
    members = df.loc[df.cluster_id == cid, "canonical_name"].unique()[:15]
    print(f"\n[{cid}] size={sz} canonical={owner!r}")
    for m in members:
        print("   ", m)

## Phase 11 : Evaluation

Score against a manually labeled ground-truth set. Targets: Precision > 98%, Recall > 95%.
Uses the capped `features_sample` from Phase 8 for the labeling worksheet.


In [ ]:
# Step 11.1 : Build labeling worksheet (label 'same'=1 / 'different'=0 by hand)
label_sample = features_sample.sample(min(1000, len(features_sample)), random_state=11)[
    ["i", "j", "owner1", "owner2", "token_set", "rapidfuzz", "levenshtein", "core_sim"]].copy()
label_sample["label"] = ""     # <-- fill manually, then save as ground_truth_labeled.csv
label_sample.to_csv(art("ground_truth_template.csv"), index=False)
print("Wrote", art("ground_truth_template.csv"))
print("Label the `label` column (1=same, 0=different), save as", art("ground_truth_labeled.csv"))

In [ ]:
# Step 11.2 : Load labeled ground truth + compute metrics (run AFTER manual labeling)
gt_path = art("ground_truth_labeled.csv")
if not os.path.exists(gt_path):
    print("No labeled file yet - complete Step 11.1 labeling first. Skipping metrics.")
else:
    gt = pd.read_csv(gt_path)
    gt = gt[gt["label"].isin([0, 1, "0", "1"])].copy()
    gt["label"] = gt["label"].astype(int)

    acc_set = set(map(tuple, accepted[["i", "j"]].values.tolist()))
    gt["pred"] = gt.apply(lambda r: int((int(r.i), int(r.j)) in acc_set), axis=1)

    tp = int(((gt.pred == 1) & (gt.label == 1)).sum())
    fp = int(((gt.pred == 1) & (gt.label == 0)).sum())
    fn = int(((gt.pred == 0) & (gt.label == 1)).sum())
    tn = int(((gt.pred == 0) & (gt.label == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    print(f"Precision : {precision:.4f}  (target > 0.98)")
    print(f"Recall    : {recall:.4f}  (target > 0.95)")
    print(f"F1        : {f1:.4f}")
    print(f"FPR       : {fp / (fp + tn) if (fp + tn) else 0.0:.4f}")
    print(f"FNR       : {fn / (fn + tp) if (fn + tp) else 0.0:.4f}")

## Phase 12 : Incremental Pipeline

Resolve a new owner without recomputing everything. Candidates come from the inverted index.


In [ ]:
# Step 12.1 : Resolve a single new owner name
def resolve_new_owner(raw_name: str):
    cn = canonicalize(normalize_typos(clean_name(raw_name)))
    cn_core = core_tokens(cn)

    cand = set()
    for tok in cn_core:
        cand.update(inverted.get(tok, ()))

    for j in cand:
        if decide(pair_features(cn, canon[j], cn_core, core_sets[j])):
            return {"canonical": cn, "cluster_id": int(df.cluster_id.iloc[j]),
                    "matched_to": canon[j], "new_cluster": False}
    return {"canonical": cn, "cluster_id": None, "matched_to": None, "new_cluster": True}

In [ ]:
# Step 12.2 : Demo (structure only)
# resolve_new_owner("D P JAIN & CO INFRASTRUCTURE PVT LTD")
# To persist: append the new row's tokens to `inverted` + core_sets, add a df row with the
# resolved/created cluster_id, re-save artifacts. No embeddings to regenerate.